# Day 10 — Counterfactual Simulation, Price Elasticity Modeling & A/B Experimentation Engine

**Project**: Google Play Store Ecosystem Analysis & NLP Autocomplete/Autocorrect Analytics  
**Internship**: Oasis Infobyte Data Analytics Internship · Task Level 2 (Task 4 & Task 5)  
**Deliverable**: Day 10 Counterfactual What-If Simulator, Econometric Price Elasticity, Bayesian A/B Testing & REST API  
**Author**: Adarsh  

---

### Objectives for Day 10:
1. **Counterfactual 'What-If' Simulator**: Quantitatively simulate changes in app parameters (APK size reduction, price discount/freemium conversion, and rating improvements) and measure predicted scale probability and revenue impacts.
2. **Empirical Price Elasticity of Demand**: Model econometric log-log elasticity ($\varepsilon_p = \frac{\partial \ln Q}{\partial \ln P}$) across catalog apps and categories to identify elastic vs. inelastic segments and revenue-maximizing price points.
3. **Frequentist & Bayesian A/B Testing Engine**: Evaluate conversion cohorts using two-sample pooled Z-tests, 95% confidence intervals, Beta-Binomial Bayesian conjugate updating, Monte Carlo superiority calculations, and expected loss.
4. **Statistical Power & Sample Size Sizing**: Formulate sample size requirements as a function of baseline conversion rates and Minimum Detectable Effects (MDE).
5. **Production Microservice & Visualization**: Expose REST endpoints (`/api/v1/simulate/what-if`, `/api/v1/experiment/ab-test`, `/api/v1/analytics/price-elasticity`) and render Figures 20 & 21.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add workspace root to path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.simulation_engine import PlayStoreSimulationEngine
engine = PlayStoreSimulationEngine(random_state=42)
print('PlayStoreSimulationEngine initialized successfully!')
print('Catalog apps loaded:', len(engine.df_apps))

## 1. Counterfactual 'What-If' Sensitivity Simulation

Studio product managers often need to evaluate trade-offs before executing costly engineering updates.  
Below we simulate the impact of optimizing APK size by **-30%** (via dynamic feature modules / App Bundles) combined with a **+0.20** user rating boost.

In [2]:
scenario = engine.simulate_what_if(
    app_name='Subway Surfers',
    delta_size_pct=-0.30,
    delta_rating=0.20
)
for k, v in scenario.to_dict().items():
    print(f'{k:30}: {v}')

## 2. Econometric Price Elasticity of Demand & Optimal Pricing Frontiers

Using log-log regressions on paid apps ($\ln(\text{Installs}) = \alpha + \varepsilon_p \ln(\text{Price}) + \epsilon$), we determine category price elasticity of demand.

In [3]:
elasticity_summary = engine.analyze_price_elasticity()
print(f"Catalog Elasticity: {elasticity_summary['overall_elasticity']:.3f} (R2: {elasticity_summary['overall_r_squared']:.3f})")
print(f"Optimal Catalog Price: ${elasticity_summary['overall_optimal_price']:.2f}")

df_benchmarks = pd.DataFrame(elasticity_summary['category_benchmarks'])
df_benchmarks.head(10)

## 3. Frequentist & Bayesian A/B Experimentation Engine

Evaluating a proposed new store icon or onboarding tutorial across 10,000 visitors per variant:
- Control Variant A: 500 conversions (5.00%)
- Treatment Variant B: 580 conversions (5.80%)

In [4]:
ab_res = engine.run_ab_experiment(
    visitors_a=10000,
    conversions_a=500,
    visitors_b=10000,
    conversions_b=580,
    alpha=0.05
)
print(f'Z-statistic: {ab_res.z_statistic:.3f} | p-value: {ab_res.p_value:.4f}')
print(f'Significant (alpha=0.05): {ab_res.is_statistically_significant}')
print(f'Bayesian P(B > A): {ab_res.bayesian_prob_b_beats_a*100:.2f}%')
print(f'Expected Loss: {ab_res.expected_loss:.6f}')
print(f'Decision: {ab_res.decision}')
print(f'Verdict: {ab_res.summary_verdict}')

## 4. Statistical Power Analysis & Sample Size Sizing Matrix

In [5]:
df_matrix = engine.generate_sample_size_matrix()
df_matrix

## 5. Artifact Generation & Visualization (Figures 20 & 21)

In [6]:
artifacts = engine.generate_reports_and_figures()
for name, path in artifacts.items():
    print(f'{name:15}: {path}')